In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix, classification_report
import tensorflow 
for gpu in tensorflow.config.experimental.list_physical_devices("GPU"):
    tensorflow.config.experimental.set_memory_growth(gpu, True)

In [ ]:
train_data = pd.read_csv('adult_trdata.csv', names=['age','workclass', 'fnlwgt', 'education', 'education-num', 'marital-status',
                                                     'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss',
                                                     'hours-per-week', 'native-country', 'income'],sep=',\s', na_values=["?"],engine='python')
print(train_data.shape)
display(train_data.head())

In [ ]:
train_data.isnull().sum()

In [ ]:
new_train = train_data.dropna()
new_train.shape

In [ ]:
new_train['workclass'].value_counts()

In [ ]:
new_train['education'].value_counts()

In [ ]:
new_train['income'].value_counts()

In [ ]:
new_train.dtypes

In [ ]:
plt.figure(figsize = (10, 10))
corr_matrix = new_train[['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']].corr()
sns.heatmap(corr_matrix, annot = True, cbar = False)

In [ ]:
new_train.dtypes

In [ ]:
# Dummy for categorical variables (training)
def dummies(x):
    dummy = pd.get_dummies(new_train[x])
    new_train.drop(columns = x, inplace = True)
    return pd.concat([new_train, dummy], axis=1)

new_train = dummies('workclass')
new_train = dummies('education')
new_train = dummies('marital-status')
new_train = dummies('occupation')
new_train = dummies('relationship')
new_train = dummies('race')
new_train = dummies('sex')
new_train = dummies('native-country')
new_train.shape

In [ ]:
new_train.head()

In [ ]:
# new_train.loc[(new_train['income'] == '>=50K') | (new_train['income'] == '>50K')] = 1
# new_train.loc[(new_train['income'] == '<=50K') | (new_train['income'] == '<50K')] = 0

new_train['income'] = new_train['income'].apply(lambda x:1 if x =='>=50K' or x =='>50K' else 0)
new_train.head()

In [ ]:
new_train.iloc[:, :6].hist(bins = 30, figsize = (15,15), color = 'b')

In [ ]:
test_dataRaw = pd.read_csv('adult_test.csv', names=['age','workclass', 'fnlwgt', 'education', 'education-num', 'marital-status',
                                                     'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss',
                                                     'hours-per-week', 'native-country', 'income'], sep=',\s', na_values=["?"],engine='python')
print(test_dataRaw.shape)
test_data = test_dataRaw.drop([0])
print(test_data.shape)
test_data = test_data.reset_index(drop=True)
display(test_data.head())

In [ ]:
test_data.isnull().sum()

In [ ]:
new_test = test_data.dropna()
new_test.shape

In [ ]:
new_test['workclass'].value_counts()

In [ ]:
new_test['education'].value_counts()

In [ ]:
new_test['income'].value_counts()

In [ ]:
# Dummy for categorical variables (testing)
def dummies_test(x):
    dummy_test = pd.get_dummies(new_test[x])
    new_test.drop(columns = x, inplace = True)
    return pd.concat([new_test, dummy_test], axis=1)

new_test = dummies_test('workclass')
new_test = dummies_test('education')
new_test = dummies_test('marital-status')
new_test = dummies_test('occupation')
new_test = dummies_test('relationship')
new_test = dummies_test('race')
new_test = dummies_test('sex')
new_test = dummies_test('native-country')
new_test.shape

In [ ]:
new_test.columns

In [ ]:
new_test['income'] = new_test['income'].apply(lambda x:1 if x =='>=50K.' or x =='>50K.' else 0)
new_test.shape

In [ ]:
## Check the column difference between train and test data
extra_column = [x for x in list(new_train.columns) if x not in list(new_test.columns)]
extra_column

In [ ]:
new_test['Holand-Netherlands'] = 0
new_test.shape

## (1). XGBOOST Classifier (Model)

In [ ]:
X_train, Y_train = np.asarray(new_train.loc[:, new_train.columns !='income']).astype(np.float32), np.asarray(new_train['income']).astype(np.float32)
X_test, Y_test = np.asarray(new_test.loc[:, new_test.columns !='income']).astype(np.float32), np.asarray(new_test['income']).astype(np.float32)

In [ ]:
X_train.shape, Y_train.shape, X_test.shape, Y_test.shape

In [ ]:
import xgboost as xgb
model1 = xgb.XGBClassifier(objective ='reg:squarederror', learning_rate = 0.1, max_depth = 5, n_estimators = 100)
model1.fit(X_train, Y_train)

In [ ]:
model1

In [ ]:
Y_pred = model1.predict(X_test)
print("Accuracy {} %".format( 100 * accuracy_score(Y_pred, Y_test)))

In [ ]:
# from xgboost import XGBClassifier
# from sklearn.model_selection import GridSearchCV
# xgb_model = XGBClassifier(learning_rate=0.01, n_estimators=100, objective='binary:logistic')
# param_grid = {'gamma': [0.2, 1, 10],  'subsample': [0.6, 0.8, 1.0], 'colsample_bytree': [0.6, 0.8, 1.0], 'max_depth': [3, 7, 10]} 
# grid = GridSearchCV(xgb_model, param_grid, refit = True, verbose = 4)
# grid.fit(X_train, Y_train)

## (2). Logistic Regression (Model)

In [ ]:
from sklearn.linear_model import LogisticRegression
model2 = LogisticRegression(random_state = 0, penalty = 'l2')
model2.fit(X_train, Y_train)
Y_pred2 = model2.predict(X_test)
print("Accuracy {} %".format( 100 * accuracy_score(Y_pred2, Y_test)))

## (3).SVM (Model)

In [ ]:
## SVM (rbf)
from sklearn.svm import SVC
model3 = SVC(random_state = 0, kernel = 'rbf')
model3.fit(X_train, Y_train)

# Predicting Test Set
y_pred3 = model3.predict(X_test)
print("Accuracy {} %".format( 100 * accuracy_score(y_pred3, Y_test)))

## (4). Random Forest (Model)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
model4 = RandomForestClassifier(n_estimators=100)
model4.fit(X_train, Y_train)

y_pred4 = model4.predict(X_test)
model4.score(X_test, Y_test)
# print("Accuracy {} %".format( 100 * accuracy_score(y_pred4, Y_test)))

In [ ]:
## Deep Nueral Network
# from tensorflow.keras import Sequential
# from tensorflow.keras.layers import Conv2D, Flatten, Dense

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
model_5 = Sequential([
    Dense(units=16, input_dim = 104,activation='relu'),
    Dense(units=24,activation='relu'),
    Dropout(0.5),
    Dense(20,activation='relu'),
    Dense(24,activation='relu'),
    Dense(1,activation='sigmoid'),
])
model_5.summary()

In [ ]:
# Train model
model_5.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_5.fit(X_train, Y_train, batch_size=20, epochs=25)
score_5 = model_5.evaluate(X_test, Y_test)
print(score_5)

## Artificial Neural Network Model (ANN)

In [ ]:
## ANN
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout

In [ ]:
# import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Dense, Activation, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam

In [ ]:
# optimizer = Adam()
ANN_model = keras.Sequential()
ANN_model.add(Dense(50, input_dim = 104))
ANN_model.add(Activation('relu'))
ANN_model.add(Dense(150))
ANN_model.add(Activation('relu'))
ANN_model.add(Dropout(0.5))
ANN_model.add(Dense(150))
ANN_model.add(Activation('relu'))
ANN_model.add(Dropout(0.3))
ANN_model.add(Dense(50))
ANN_model.add(Activation('linear'))
ANN_model.add(Dense(1))
ANN_model.compile(loss = 'mse', optimizer = 'adam')
ANN_model.summary()

In [ ]:
ANN_model.compile(optimizer='Adam', loss='mean_squared_error')
epochs_hist = ANN_model.fit(X_train, Y_train, epochs = 50, batch_size = 20, validation_split = 0.2)

In [ ]:
result = ANN_model.evaluate(X_test, Y_test)
accuracy_ANN = 1 - result
print("Accuracy : {}".format(accuracy_ANN))